# 5 - Exploring models

Here the goal is not to crown a universal best model. The goal is to show that:

- different model families can all be used inside the same `mltsa` workflow
- some models expose **native** importance directly
- others need **perturbation-based** explanations such as permutation or global mean

To keep the comparison fair and fast, we use the same flattened early-time window for
every model.

In [ ]:
from pathlib import Path
import sys

try:
    import mltsa  # noqa: F401
except ImportError:
    for parent in (Path.cwd(), *Path.cwd().parents):
        src_dir = parent / "src"
        if (src_dir / "mltsa").exists():
            sys.path.insert(0, str(src_dir))
            break
    import mltsa  # noqa: F401

import numpy as np
import matplotlib.pyplot as plt

for parent in (Path.cwd(), *Path.cwd().parents):
    notebooks_dir = parent / "notebooks"
    if notebooks_dir.exists():
        DATA_DIR = notebooks_dir / "_generated"
        break
else:
    DATA_DIR = Path.cwd() / "_generated"

DATA_DIR.mkdir(exist_ok=True, parents=True)
plt.style.use("seaborn-v0_8-whitegrid")
np.set_printoptions(precision=3, suppress=True)

In [ ]:
from sklearn.model_selection import train_test_split

from mltsa.explain import analyze
from mltsa.models import get_model
from mltsa.synthetic import make_1d_dataset

dataset = make_1d_dataset(
    n_trajectories=200,  # Quick comparison dataset
    n_steps=64,  # Total steps per trajectory
    n_features=10,  # Total observed features
    n_relevant=3,  # Hidden ground-truth relevant features
    base_seed=2024,  # Reproducible dataset definition
)

window = 28  # Shared early-time window for all models
X = dataset.X[:, :window, :].reshape(dataset.n_trajectories, -1)
y = dataset.y
flat_feature_names = tuple(
    f"{name}@t{step:03d}"
    for step in range(window)
    for name in dataset.feature_names
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=0,
    stratify=y,
)

We compare four models:

- `RandomForest` and `GradientBoosting` from sklearn
- `MLP` and `CNN1D` from the lightweight torch wrappers

The tree models expose native importances. The neural models do not, so we switch to
`global_mean` and `permutation`.

In [ ]:
model_specs = [
    ("RandomForest", "random_forest", {"n_estimators": 120}, "native", {}),
    ("GradientBoosting", "gradient_boosting", {"n_estimators": 120}, "native", {}),
    ("Torch MLP", "mlp", {"epochs": 10, "hidden_sizes": (128, 64), "batch_size": 32}, "global_mean", {}),
    ("Torch CNN1D", "cnn1d", {"epochs": 10, "channels": 24, "batch_size": 32}, "permutation", {"n_repeats": 3, "n_jobs": 1}),
]

summaries = []
explanations = {}

for label, model_name, kwargs, method, extra_kwargs in model_specs:
    model = get_model(model_name, **kwargs)
    model.fit(X_train, y_train)
    score = model.score(X_test, y_test)
    explanation = analyze(
        model,
        method=method,
        X=X_test,
        y=y_test,
        feature_names=flat_feature_names,
        **extra_kwargs,
    )

    base_importance = explanation.importances.reshape(window, dataset.n_features).mean(axis=0)
    explanations[label] = base_importance
    summaries.append((label, method, score))

for label, method, score in summaries:
    print(f"{label:18s} | method={method:11s} | test_accuracy={score:.3f}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharey=True)
axes = axes.ravel()

for axis, (label, base_importance) in zip(axes, explanations.items()):
    colors = [
        "#e76f51" if index in dataset.relevant_features else "#c7ced6"
        for index in range(dataset.n_features)
    ]
    axis.bar(np.arange(dataset.n_features), base_importance, color=colors)
    axis.set_title(label)
    axis.set_xticks(np.arange(dataset.n_features), dataset.feature_names, rotation=45, ha="right")
    axis.set_ylabel("mean importance")

plt.tight_layout()
plt.show()

A nice rule of thumb:

- **tree models**: try native importance first
- **neural models**: expect to use perturbation methods more often